# Transfer Learning — DistilBERT

**Kaggle, GPU required (T4 x2 — P100 is still broken on Kaggle's default image, same issue as the BiLSTM notebooks).** Fine-tunes `distilbert-base-uncased` on the same train/val/test splits used throughout this project, with a manual PyTorch training loop (not HF `Trainer`) to keep the code style consistent with the BiLSTM notebooks and to reuse `evaluate_predictions`/`print_metrics` unchanged.

**Deliberately different from the BiLSTM setup, worth knowing before running:**
- **`MAX_LEN` is capped at 512** — DistilBERT uses learned absolute positional embeddings (a fixed 512-row lookup table baked in at pretraining time), unlike the BiLSTM's recurrence which has no such limit. This isn't a tunable choice the way MAX_LEN=1024 was.
- **Learning rate 2e-5, not 1e-3.** Fine-tuning a pretrained transformer needs a much smaller LR than training an embedding+LSTM from scratch — the BiLSTM's LR would destroy the pretrained weights almost immediately.
- **2-4 epochs expected, not 10-18.** Pretrained weights mean far less training is needed before overfitting sets in.
- **Mixed precision (AMP) is used here**, unlike the BiLSTM notebooks. This actually matters for a Transformer's dense attention matmuls on a T4's Tensor Cores, unlike the BiLSTM case where we found AMP wouldn't have helped without it already being on.

Not wired into the repo/git — standalone for now, per your request.

In [ ]:
import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import transformers
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "transformers"], check=True)
    import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

print(f"transformers version: {transformers.__version__}")
print(f"torch version: {torch.__version__}")


### Environment setup (Kaggle)

In [ ]:
import platform
import socket

IS_KAGGLE = os.path.exists("/kaggle/input")

print("=== ENVIRONMENT CHECK ===")
print(f"Hostname:        {socket.gethostname()}")
print(f"Platform:        {platform.system()} {platform.machine()}")
print(f"IS_KAGGLE:       {IS_KAGGLE}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected — fine-tuning DistilBERT on CPU will be very slow. "
          "Check Settings -> Accelerator -> GPU T4 x2.")

if IS_KAGGLE:
    print("\n=== /kaggle/input contents ===")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth > 2:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{root}/")
        for fname in files:
            fpath = os.path.join(root, fname)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"{indent}  {fname}  ({size_mb:.1f} MB)")


def find_kaggle_file(filename, search_root="/kaggle/input"):
    """Search recursively for a file by exact name under search_root."""
    matches = []
    for root, dirs, files in os.walk(search_root):
        if filename in files:
            matches.append(os.path.join(root, filename))
    if not matches:
        return None
    if len(matches) > 1:
        print(f"WARNING: multiple matches for '{filename}': {matches} — using first")
    return matches[0]


### Config

In [ ]:
SEED = 42

MODEL_NAME = "distilbert-base-uncased"

# Data
TEXT_COL = "text"
LABEL_COL = "label"

# Sequence — 512 is a hard architectural ceiling for DistilBERT, not a free choice
MAX_LEN = None  # locked below after checking real coverage at the 512 cap

# Training — standard transformer fine-tuning defaults, deliberately different
# from the BiLSTM's from-scratch training config
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_EPOCHS = 4
PATIENCE = 2
GRAD_CLIP = 1.0
USE_AMP = True   # mixed precision — genuinely useful here, unlike the BiLSTM case

# Artifacts
SAVE_DIR = "/kaggle/working/saved/distilbert" if IS_KAGGLE else "saved/distilbert"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
# ------------------------------------------------------------------


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model: {MODEL_NAME}")
print(f"AMP enabled: {AMP_ENABLED}")
print(f"Save dir: {SAVE_DIR}")


In [ ]:
if IS_KAGGLE:
    TRAIN_PATH = find_kaggle_file("train.csv")
    VAL_PATH   = find_kaggle_file("val.csv")
    TEST_PATH  = find_kaggle_file("test.csv")
    for _name, _path in [("train.csv", TRAIN_PATH), ("val.csv", VAL_PATH), ("test.csv", TEST_PATH)]:
        if _path is None:
            raise FileNotFoundError(f"{_name} not found under /kaggle/input — check dataset is attached.")
else:
    TRAIN_PATH = "../data/processed/train.csv"
    VAL_PATH   = "../data/processed/val.csv"
    TEST_PATH  = "../data/processed/test.csv"


def load_split(path, text_col, label_col):
    frame = pd.read_csv(path)
    frame[text_col] = frame[text_col].fillna("").astype(str)
    frame[label_col] = frame[label_col].astype(int)
    return frame


train_df = load_split(TRAIN_PATH, TEXT_COL, LABEL_COL)
val_df   = load_split(VAL_PATH,   TEXT_COL, LABEL_COL)
test_df  = load_split(TEST_PATH,  TEXT_COL, LABEL_COL)

for name, frame in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name:<6} {len(frame):>7,} rows | {frame[LABEL_COL].mean():.4f} positive")


### Tokenizer & sequence length
DistilBERT's WordPiece tokenizer produces a different length distribution than the BiLSTM's word-level tokenizer — re-checking coverage here rather than assuming the earlier analysis carries over. Unlike the BiLSTM, there's no length to *choose* past 512 — this only tells you how much gets cut off.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Real token counts with NO truncation, to see the true distribution
train_encodings_full = tokenizer(list(train_df[TEXT_COL]), truncation=False)
train_lengths = np.array([len(ids) for ids in train_encodings_full["input_ids"]])

percentiles = [50, 75, 90, 95, 97, 99]
print("WordPiece token-length percentiles (train set, DistilBERT tokenizer):")
for p in percentiles:
    print(f"  p{p:>2}: {int(np.percentile(train_lengths, p)):>6,}")
print(f"  max: {train_lengths.max():>6,}")
print(f" mean: {train_lengths.mean():>6.1f}")

del train_encodings_full  # free memory — only needed the lengths


In [ ]:
MAX_LEN = 512  # hard architectural ceiling for DistilBERT — not a free choice

coverage_overall = (train_lengths <= MAX_LEN).mean()
coverage_benign = (train_lengths[train_df[LABEL_COL].values == 0] <= MAX_LEN).mean()
coverage_malicious = (train_lengths[train_df[LABEL_COL].values == 1] <= MAX_LEN).mean()

print(f"MAX_LEN = {MAX_LEN}")
print(f"Overall coverage:   {coverage_overall:.2%}")
print(f"Benign coverage:    {coverage_benign:.2%}")
print(f"Malicious coverage: {coverage_malicious:.2%}")


### Dataset & DataLoader (dynamic padding)
Sequences are tokenized to variable length (no fixed padding here) and padded per-batch by `DataCollatorWithPadding` — batches of short prompts process faster than always padding to 512, unlike the BiLSTM's fixed-length approach.

In [ ]:
class TransformerPromptDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_ds = TransformerPromptDataset(train_df[TEXT_COL], train_df[LABEL_COL], tokenizer, MAX_LEN)
val_ds   = TransformerPromptDataset(val_df[TEXT_COL],   val_df[LABEL_COL],   tokenizer, MAX_LEN)
test_ds  = TransformerPromptDataset(test_df[TEXT_COL],  test_df[LABEL_COL],  tokenizer, MAX_LEN)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator, collate_fn=data_collator)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")


### Model

In [ ]:
# num_labels=1: single logit output, matching the BiLSTM's binary setup exactly.
# HF's own internal loss computation is bypassed — labels are never passed into
# the model call, loss is computed manually below with BCEWithLogitsLoss +
# pos_weight, same as every other model in this project.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model on device: {next(model.parameters()).device}")


### Loss, optimizer, scheduler

In [ ]:
num_neg = int((train_df[LABEL_COL] == 0).sum())
num_pos = int((train_df[LABEL_COL] == 1).sum())
pos_weight_value = num_neg / num_pos

print(f"Negatives: {num_neg:,} | Positives: {num_pos:,}")
print(f"pos_weight = {pos_weight_value:.4f}")

pos_weight = torch.tensor([pos_weight_value], dtype=torch.float, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

print(f"Total optimizer steps: {total_steps:,} (warmup: {warmup_steps:,})")


### Train and evaluate functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device, grad_clip, epoch_num, total_epochs, print_every=40):
    model.train()
    running_loss = 0.0
    n_batches = len(loader)
    batch_start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze(-1)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * input_ids.size(0)

        if batch_idx == 1 or batch_idx % print_every == 0 or batch_idx == n_batches:
            elapsed = time.time() - batch_start
            avg_batch_time = elapsed / batch_idx
            eta_sec = avg_batch_time * (n_batches - batch_idx)
            print(
                f"  [Epoch {epoch_num}/{total_epochs}] batch {batch_idx}/{n_batches} "
                f"| loss {loss.item():.4f} | {avg_batch_time:.2f}s/batch | ETA {eta_sec:.0f}s",
                flush=True,
            )

    return running_loss / len(loader.dataset)


def predict(model, loader, criterion, device, desc="eval"):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits.squeeze(-1)
                loss = criterion(logits, labels)

            running_loss += loss.item() * input_ids.size(0)

            probs = torch.sigmoid(logits.float())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, np.concatenate(all_labels), np.concatenate(all_probs)


### Helper functions

In [ ]:
def evaluate_predictions(y_true, y_prob, threshold=0.5, split_name=""):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "split": split_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }


def print_metrics(metrics):
    print(f"--- {metrics['split']} (threshold={metrics['threshold']:.2f}) ---")
    for key in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
        print(f"  {key:<10}: {metrics[key]:.4f}")


Quick timing check before committing to the full run — same pattern used throughout this project. Given the uncertainty flagged earlier about per-model training cost, don't skip this one.

In [ ]:
model.train()
single_batch = next(iter(train_loader))
input_ids = single_batch["input_ids"].to(DEVICE)
attention_mask = single_batch["attention_mask"].to(DEVICE)
labels = single_batch["labels"].to(DEVICE)

start = time.time()
optimizer.zero_grad()
with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits.squeeze(-1)
    loss = criterion(logits, labels)
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
scaler.step(optimizer)
scaler.update()
scheduler.step()
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
batch_time = time.time() - start

print(f"Single batch time ({DEVICE}, AMP={AMP_ENABLED}, MAX_LEN={MAX_LEN}, batch={BATCH_SIZE}): {batch_time:.3f}s")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Rough epoch estimate: {batch_time * len(train_loader) / 60:.1f} min")
print(f"Rough {MAX_EPOCHS}-epoch estimate: {batch_time * len(train_loader) * MAX_EPOCHS / 60:.1f} min")
print("\nNote: this ran one real optimizer step, so training below starts from a")
print("slightly-updated model rather than the pristine pretrained checkpoint —")
print("negligible effect on results, same tradeoff accepted in every prior notebook.")


### Training loop

In [ ]:
best_val_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0
history = []

checkpoint_dir = os.path.join(SAVE_DIR, "distilbert_best")

print(f"Starting training — up to {MAX_EPOCHS} epochs, early stopping patience={PATIENCE}\n")

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()

    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler,
        DEVICE, GRAD_CLIP, epoch, MAX_EPOCHS
    )
    val_loss, val_true, val_prob = predict(model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch}/{MAX_EPOCHS} [val]")
    val_metrics = evaluate_predictions(val_true, val_prob, 0.5, "val")

    elapsed = time.time() - start

    history.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_f1": val_metrics["f1"], "val_recall": val_metrics["recall"],
        "val_precision": val_metrics["precision"], "seconds": elapsed,
    })

    print(
        f"Epoch {epoch:>2}/{MAX_EPOCHS} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
        f"| val_F1 {val_metrics['f1']:.4f} | val_recall {val_metrics['recall']:.4f} | {elapsed:.1f}s"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        model.save_pretrained(checkpoint_dir)
        tokenizer.save_pretrained(checkpoint_dir)
        print("         -> new best val F1, checkpoint saved")
    else:
        epochs_without_improvement += 1
        print(f"         -> no improvement ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}, best val F1: {best_val_f1:.4f}.")
            break

history_df = pd.DataFrame(history)
print(f"\nTraining complete. Best val F1 = {best_val_f1:.4f} at epoch {best_epoch}")


### Load checkpoint and evaluate

In [1]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(DEVICE)

_, val_true, val_prob = predict(model, val_loader, criterion, DEVICE, desc="final val")
val_metrics = evaluate_predictions(val_true, val_prob, 0.5, "val")
print_metrics(val_metrics)

_, test_true, test_prob = predict(model, test_loader, criterion, DEVICE, desc="final test")
test_metrics = evaluate_predictions(test_true, test_prob, 0.5, "test")
print_metrics(test_metrics)

print("\nConfusion matrix (test, threshold=0.5):")
test_pred = (test_prob >= 0.5).astype(int)
cm = confusion_matrix(test_true, test_pred)
print(pd.DataFrame(cm, index=["actual_benign", "actual_malicious"], columns=["pred_benign", "pred_malicious"]))

print("\n" + classification_report(test_true, test_pred, digits=4))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_58/1074905201.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


--- val (threshold=0.50) ---
  accuracy  : 0.8655
  precision : 0.8628
  recall    : 0.7574
  f1        : 0.8067
  roc_auc   : 0.9294
  pr_auc    : 0.9115


/tmp/ipykernel_58/1074905201.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


--- test (threshold=0.50) ---
  accuracy  : 0.8708
  precision : 0.8680
  recall    : 0.7681
  f1        : 0.8150
  roc_auc   : 0.9269
  pr_auc    : 0.9103

Confusion matrix (test, threshold=0.5):
                  pred_benign  pred_malicious
actual_benign            1814             134
actual_malicious          266             881

              precision    recall  f1-score   support

         0.0     0.8721    0.9312    0.9007      1948
         1.0     0.8680    0.7681    0.8150      1147

    accuracy                         0.8708      3095
   macro avg     0.8700    0.8497    0.8578      3095
weighted avg     0.8706    0.8708    0.8689      3095



### Save metrics/history

In [ ]:
config = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "grad_clip": GRAD_CLIP,
    "amp_enabled": AMP_ENABLED,
    "pos_weight": pos_weight_value,
    "best_epoch": best_epoch,
    "best_val_f1": best_val_f1,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "device": str(DEVICE),
}
with open(os.path.join(SAVE_DIR, "config_distilbert.json"), "w") as f:
    json.dump(config, f, indent=2)

results = pd.DataFrame([val_metrics, test_metrics])
results.to_csv(os.path.join(SAVE_DIR, "metrics_distilbert.csv"), index=False)
history_df.to_csv(os.path.join(SAVE_DIR, "history_distilbert.csv"), index=False)

print("Saved:")
for root, dirs, files in os.walk(SAVE_DIR):
    for f in files:
        print(f"  {os.path.join(root, f)}")

results


Zip artifacts for download

In [ ]:
import shutil

zip_path = shutil.make_archive(
    base_name=f"{SAVE_DIR}/../distilbert_artifacts",
    format="zip",
    root_dir=SAVE_DIR,
)
print(f"Zipped to: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1e6:.2f} MB")


In [2]:
def sweep_thresholds(y_true, y_prob, thresholds):
    rows = []
    for t in thresholds:
        rows.append(evaluate_predictions(y_true, y_prob, threshold=t, split_name="val"))
    return pd.DataFrame(rows)


thresholds = np.arange(0.05, 0.96, 0.05)
threshold_sweep_df = sweep_thresholds(val_true, val_prob, thresholds)

pd.set_option("display.width", 160)
print(threshold_sweep_df[["threshold", "accuracy", "precision", "recall", "f1"]].round(4).to_string(index=False))

 threshold  accuracy  precision  recall     f1
      0.05    0.7456     0.6004  0.9363 0.7317
      0.10    0.8012     0.6727  0.9023 0.7708
      0.15    0.8297     0.7193  0.8857 0.7939
      0.20    0.8429     0.7481  0.8682 0.8037
      0.25    0.8510     0.7699  0.8525 0.8091
      0.30    0.8539     0.7840  0.8360 0.8091
      0.35    0.8565     0.7955  0.8246 0.8098
      0.40    0.8588     0.8069  0.8133 0.8101
      0.45    0.8623     0.8191  0.8063 0.8127
      0.50    0.8655     0.8628  0.7574 0.8067
      0.55    0.8668     0.8730  0.7496 0.8066
      0.60    0.8678     0.8819  0.7426 0.8063
      0.65    0.8668     0.8913  0.7295 0.8023
      0.70    0.8659     0.9003  0.7173 0.7984
      0.75    0.8649     0.9072  0.7077 0.7951
      0.80    0.8652     0.9175  0.6990 0.7935
      0.85    0.8623     0.9235  0.6850 0.7866
      0.90    0.8610     0.9377  0.6693 0.7811
      0.95    0.8571     0.9536  0.6457 0.7700


In [3]:
def find_threshold_for_recall(sweep_df, target_recall):
    """Highest threshold that still clears target_recall (higher threshold = better precision)."""
    candidates = sweep_df[sweep_df["recall"] >= target_recall]
    if candidates.empty:
        return None
    return candidates.sort_values("threshold", ascending=False).iloc[0]


for target in [0.75, 0.80, 0.8056, 0.85, 0.90]:
    row = find_threshold_for_recall(threshold_sweep_df, target)
    if row is not None:
        print(f"Target recall >= {target:.4f}: threshold={row['threshold']:.2f} "
              f"-> val recall={row['recall']:.4f}, precision={row['precision']:.4f}, f1={row['f1']:.4f}")
    else:
        print(f"Target recall >= {target:.4f}: not achievable in this sweep range (try lower thresholds)")

Target recall >= 0.7500: threshold=0.50 -> val recall=0.7574, precision=0.8628, f1=0.8067
Target recall >= 0.8000: threshold=0.45 -> val recall=0.8063, precision=0.8191, f1=0.8127
Target recall >= 0.8056: threshold=0.45 -> val recall=0.8063, precision=0.8191, f1=0.8127
Target recall >= 0.8500: threshold=0.25 -> val recall=0.8525, precision=0.7699, f1=0.8091
Target recall >= 0.9000: threshold=0.10 -> val recall=0.9023, precision=0.6727, f1=0.7708


In [5]:
candidate_thresholds = [0.50, 0.45, 0.25]

comparison_rows = []
for t in candidate_thresholds:
    m = evaluate_predictions(test_true, test_prob, threshold=t, split_name=f"test @ {t}")
    comparison_rows.append(m)

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df[["split", "threshold", "accuracy", "precision", "recall", "f1"]].round(4).to_string(index=False))

print("\nFor reference — best BiLSTM (GloVe 100d/1024, T4): recall=0.8056, precision=0.7130, f1=0.7564, FN=223")

for t in candidate_thresholds:
    pred = (test_prob >= t).astype(int)
    fn = int(((test_true == 1) & (pred == 0)).sum())
    fp = int(((test_true == 0) & (pred == 1)).sum())
    print(f"threshold={t}: FN={fn}, FP={fp}")

      split  threshold  accuracy  precision  recall     f1
 test @ 0.5       0.50    0.8708     0.8680  0.7681 0.8150
test @ 0.45       0.45    0.8578     0.8115  0.8030 0.8072
test @ 0.25       0.25    0.8414     0.7562  0.8439 0.7977

For reference — best BiLSTM (GloVe 100d/1024, T4): recall=0.8056, precision=0.7130, f1=0.7564, FN=223
threshold=0.5: FN=266, FP=134
threshold=0.45: FN=226, FP=214
threshold=0.25: FN=179, FP=312


In [1]:
import unicodedata

def normalize_text(text):
    return unicodedata.normalize("NFKC", text)


val_df_norm = val_df.copy()
val_df_norm[TEXT_COL] = val_df_norm[TEXT_COL].apply(normalize_text)

test_df_norm = test_df.copy()
test_df_norm[TEXT_COL] = test_df_norm[TEXT_COL].apply(normalize_text)

n_changed_val = (val_df[TEXT_COL] != val_df_norm[TEXT_COL]).sum()
n_changed_test = (test_df[TEXT_COL] != test_df_norm[TEXT_COL]).sum()
print(f"Val rows changed by NFKC normalization:  {n_changed_val:,} / {len(val_df):,}")
print(f"Test rows changed by NFKC normalization: {n_changed_test:,} / {len(test_df):,}")

Val rows changed by NFKC normalization:  186 / 3,094
Test rows changed by NFKC normalization: 190 / 3,095


In [2]:
val_ds_norm  = TransformerPromptDataset(val_df_norm[TEXT_COL],  val_df_norm[LABEL_COL],  tokenizer, MAX_LEN)
test_ds_norm = TransformerPromptDataset(test_df_norm[TEXT_COL], test_df_norm[LABEL_COL], tokenizer, MAX_LEN)

val_loader_norm  = DataLoader(val_ds_norm,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader_norm = DataLoader(test_ds_norm, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

_, val_true_norm, val_prob_norm = predict(model, val_loader_norm, criterion, DEVICE, desc="val (NFKC)")
_, test_true_norm, test_prob_norm = predict(model, test_loader_norm, criterion, DEVICE, desc="test (NFKC)")

/tmp/ipykernel_58/1074905201.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


In [3]:
thresholds_to_check = [0.5, 0.3, 0.25]

comparison_rows = []
for t in thresholds_to_check:
    comparison_rows.append(evaluate_predictions(test_true, test_prob, threshold=t, split_name=f"original @ {t}"))
    comparison_rows.append(evaluate_predictions(test_true_norm, test_prob_norm, threshold=t, split_name=f"NFKC-normalized @ {t}"))

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option("display.width", 160)
print(comparison_df[["split", "threshold", "accuracy", "precision", "recall", "f1"]].round(4).to_string(index=False))

print()
for t in thresholds_to_check:
    pred_orig = (test_prob >= t).astype(int)
    pred_norm = (test_prob_norm >= t).astype(int)
    fn_orig = int(((test_true == 1) & (pred_orig == 0)).sum())
    fn_norm = int(((test_true_norm == 1) & (pred_norm == 0)).sum())
    fp_orig = int(((test_true == 0) & (pred_orig == 1)).sum())
    fp_norm = int(((test_true_norm == 0) & (pred_norm == 1)).sum())
    print(f"threshold={t}: original FN={fn_orig}/FP={fp_orig}  |  NFKC FN={fn_norm}/FP={fp_norm}")

                 split  threshold  accuracy  precision  recall     f1
        original @ 0.5       0.50    0.8708     0.8680  0.7681 0.8150
 NFKC-normalized @ 0.5       0.50    0.8711     0.8674  0.7698 0.8157
        original @ 0.3       0.30    0.8456     0.7691  0.8335 0.8000
 NFKC-normalized @ 0.3       0.30    0.8468     0.7707  0.8352 0.8017
       original @ 0.25       0.25    0.8414     0.7562  0.8439 0.7977
NFKC-normalized @ 0.25       0.25    0.8423     0.7576  0.8448 0.7988

threshold=0.5: original FN=266/FP=134  |  NFKC FN=264/FP=135
threshold=0.3: original FN=191/FP=287  |  NFKC FN=189/FP=285
threshold=0.25: original FN=179/FP=312  |  NFKC FN=178/FP=310


In [4]:
if IS_KAGGLE:
    COMBINED_PATH = find_kaggle_file("combined_full.csv")
    if COMBINED_PATH is None:
        raise FileNotFoundError("combined_full.csv not found under /kaggle/input — check dataset is attached.")
else:
    COMBINED_PATH = "../data/processed/combined_full.csv"

combined_df = pd.read_csv(COMBINED_PATH)
SOURCE_COL = "source_dataset"
source_lookup = combined_df[[TEXT_COL, SOURCE_COL]].drop_duplicates(subset=TEXT_COL)


def merge_source_idempotent(df, source_lookup, source_col, text_col):
    cols_to_drop = [c for c in df.columns if c == source_col or c.startswith(f"{source_col}_")]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    merged = df.merge(source_lookup, on=text_col, how="left")
    assert len(merged) == len(df), "Merge changed row count — check for duplicate text in combined_df"
    return merged


test_df_with_source = merge_source_idempotent(test_df, source_lookup, SOURCE_COL, TEXT_COL)
print(f"Unmatched: {test_df_with_source[SOURCE_COL].isna().sum()} / {len(test_df_with_source)}")
print(test_df_with_source[SOURCE_COL].value_counts())

Unmatched: 0 / 3095
source_dataset
malicious_prompts    1504
wildjailbreak        1489
deepset               102
Name: count, dtype: int64


In [5]:
def source_recall_breakdown(test_df_with_source, test_true, test_prob, threshold, label):
    test_pred = (test_prob >= threshold).astype(int)
    rows = []
    for source in test_df_with_source[SOURCE_COL].dropna().unique():
        mask = (test_df_with_source[SOURCE_COL] == source).to_numpy()
        malicious_mask = mask & (test_true == 1)
        benign_mask = mask & (test_true == 0)

        n_malicious = int(malicious_mask.sum())
        n_benign = int(benign_mask.sum())

        recall = test_pred[malicious_mask].mean() if n_malicious > 0 else float("nan")
        fpr = test_pred[benign_mask].mean() if n_benign > 0 else float("nan")
        fn = int((malicious_mask & (test_pred == 0)).sum())
        fp = int((benign_mask & (test_pred == 1)).sum())

        rows.append({
            "config": label, "threshold": threshold, "source": source,
            "n_malicious": n_malicious, "recall": recall, "fn": fn,
            "n_benign": n_benign, "fpr": fpr, "fp": fp,
        })
    return rows


all_rows = []
for t in [0.5, 0.3, 0.25]:
    all_rows.extend(source_recall_breakdown(test_df_with_source, test_true, test_prob, t, "original"))
    all_rows.extend(source_recall_breakdown(test_df_with_source, test_true_norm, test_prob_norm, t, "NFKC"))

breakdown_df = pd.DataFrame(all_rows).sort_values(["source", "threshold", "config"]).reset_index(drop=True)
pd.set_option("display.width", 200)
print(breakdown_df[["config", "threshold", "source", "n_malicious", "recall", "fn", "n_benign", "fpr", "fp"]].round(4).to_string(index=False))

  config  threshold            source  n_malicious  recall  fn  n_benign    fpr  fp
    NFKC       0.25           deepset           38  0.9211   3        64 0.0156   1
original       0.25           deepset           38  0.9211   3        64 0.0156   1
    NFKC       0.30           deepset           38  0.9211   3        64 0.0156   1
original       0.30           deepset           38  0.9211   3        64 0.0156   1
    NFKC       0.50           deepset           38  0.9211   3        64 0.0000   0
original       0.50           deepset           38  0.9211   3        64 0.0000   0
    NFKC       0.25 malicious_prompts          373  0.6542 129      1131 0.2246 254
original       0.25 malicious_prompts          373  0.6515 130      1131 0.2263 256
    NFKC       0.30 malicious_prompts          373  0.6300 138      1131 0.2025 229
original       0.30 malicious_prompts          373  0.6247 140      1131 0.2042 231
    NFKC       0.50 malicious_prompts          373  0.4450 207      1131 0.0

In [6]:
pivot_recall = breakdown_df[breakdown_df["config"] == "original"].pivot(index="source", columns="threshold", values="recall")
print("\nRecall by source x threshold (original text):")
print(pivot_recall.round(4))


Recall by source x threshold (original text):
threshold            0.25    0.30    0.50
source                                   
deepset            0.9211  0.9211  0.9211
malicious_prompts  0.6515  0.6247  0.4397
wildjailbreak      0.9375  0.9348  0.9266


In [7]:
nfkc_fn = breakdown_df.pivot_table(index=["source", "threshold"], columns="config", values="fn").reset_index()
nfkc_fn["fn_reduction"] = nfkc_fn["original"] - nfkc_fn["NFKC"]
print("\nFN reduction from NFKC normalization, by source:")
print(nfkc_fn.sort_values(["source", "threshold"]).round(4).to_string(index=False))


FN reduction from NFKC normalization, by source:
           source  threshold  NFKC  original  fn_reduction
          deepset       0.25   3.0       3.0           0.0
          deepset       0.30   3.0       3.0           0.0
          deepset       0.50   3.0       3.0           0.0
malicious_prompts       0.25 129.0     130.0           1.0
malicious_prompts       0.30 138.0     140.0           2.0
malicious_prompts       0.50 207.0     209.0           2.0
    wildjailbreak       0.25  46.0      46.0           0.0
    wildjailbreak       0.30  48.0      48.0           0.0
    wildjailbreak       0.50  54.0      54.0           0.0


In [8]:
model_baseline = model  # keep a reference to the original fine-tuned model
test_prob_baseline = test_prob.copy()
test_true_baseline = test_true.copy()
val_prob_baseline = val_prob.copy()
val_true_baseline = val_true.copy()

print("Baseline DistilBERT results preserved for comparison.")

Baseline DistilBERT results preserved for comparison.


In [9]:
def merge_source_idempotent(df, source_lookup, source_col, text_col):
    cols_to_drop = [c for c in df.columns if c == source_col or c.startswith(f"{source_col}_")]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    merged = df.merge(source_lookup, on=text_col, how="left")
    assert len(merged) == len(df), "Merge changed row count — check for duplicate text in combined_df"
    return merged


def build_oversampled_train_df(train_with_source, target_source, oversample_factor, source_col, label_col):
    target_mask = (train_with_source[source_col] == target_source) & (train_with_source[label_col] == 1)
    target_rows = train_with_source[target_mask]
    extra_copies = pd.concat([target_rows] * (oversample_factor - 1), ignore_index=True)
    oversampled = pd.concat([train_with_source, extra_copies], ignore_index=True)
    return oversampled.drop(columns=[source_col]).reset_index(drop=True), len(target_rows)


train_df_with_source = merge_source_idempotent(train_df, source_lookup, SOURCE_COL, TEXT_COL)

OVERSAMPLE_FACTOR = 2
train_df_oversampled, n_oversampled = build_oversampled_train_df(
    train_df_with_source, target_source="malicious_prompts",
    oversample_factor=OVERSAMPLE_FACTOR, source_col=SOURCE_COL, label_col=LABEL_COL,
)

print(f"Original train:    {len(train_df):,} rows | positive rate {train_df[LABEL_COL].mean():.4f}")
print(f"Oversampled train: {len(train_df_oversampled):,} rows | positive rate {train_df_oversampled[LABEL_COL].mean():.4f} "
      f"({n_oversampled:,} malicious_prompts malicious rows duplicated {OVERSAMPLE_FACTOR}x)")

Original train:    14,441 rows | positive rate 0.3706
Oversampled train: 16,032 rows | positive rate 0.4331 (1,591 malicious_prompts malicious rows duplicated 2x)


In [10]:
train_ds_oversampled = TransformerPromptDataset(
    train_df_oversampled[TEXT_COL], train_df_oversampled[LABEL_COL], tokenizer, MAX_LEN
)

generator_os = torch.Generator()
generator_os.manual_seed(SEED)
train_loader_oversampled = DataLoader(
    train_ds_oversampled, batch_size=BATCH_SIZE, shuffle=True,
    generator=generator_os, collate_fn=data_collator,
)

print(f"Batches — train (oversampled): {len(train_loader_oversampled)}, val: {len(val_loader)}, test: {len(test_loader)}")

Batches — train (oversampled): 1002, val: 194, test: 194


In [11]:
set_seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEVICE)

num_neg_os = int((train_df_oversampled[LABEL_COL] == 0).sum())
num_pos_os = int((train_df_oversampled[LABEL_COL] == 1).sum())
pos_weight_value_os = num_neg_os / num_pos_os
pos_weight_os = torch.tensor([pos_weight_value_os], dtype=torch.float, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_os)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader_oversampled) * MAX_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

print(f"pos_weight (oversampled): {pos_weight_value_os:.4f}")
print(f"Total optimizer steps: {total_steps:,} (warmup: {warmup_steps:,})")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


pos_weight (oversampled): 1.3091
Total optimizer steps: 4,008 (warmup: 400)


/tmp/ipykernel_58/1230481311.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


In [12]:
best_val_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0
history_oversampled = []

checkpoint_dir_oversampled = os.path.join(SAVE_DIR, f"distilbert_oversampled{OVERSAMPLE_FACTOR}x_best")

print(f"Starting training — up to {MAX_EPOCHS} epochs, early stopping patience={PATIENCE}\n")

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(
        model, train_loader_oversampled, criterion, optimizer, scheduler, scaler,
        DEVICE, GRAD_CLIP, epoch, MAX_EPOCHS
    )
    val_loss, val_true_os, val_prob_os = predict(model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch} val")
    val_metrics_os = evaluate_predictions(val_true_os, val_prob_os, 0.5, "val")

    elapsed = time.time() - start
    history_oversampled.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_f1": val_metrics_os["f1"], "val_recall": val_metrics_os["recall"], "seconds": elapsed,
    })

    print(f"Epoch {epoch:>2}/{MAX_EPOCHS} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
          f"| val_F1 {val_metrics_os['f1']:.4f} | val_recall {val_metrics_os['recall']:.4f} | {elapsed:.1f}s")

    if val_metrics_os["f1"] > best_val_f1:
        best_val_f1 = val_metrics_os["f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        model.save_pretrained(checkpoint_dir_oversampled)
        tokenizer.save_pretrained(checkpoint_dir_oversampled)
        print("         -> new best, checkpoint saved")
    else:
        epochs_without_improvement += 1
        print(f"         -> no improvement ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best epoch {best_epoch}.")
            break

print(f"\nTraining complete. Best val F1 = {best_val_f1:.4f} at epoch {best_epoch}")

Starting training — up to 4 epochs, early stopping patience=2



/tmp/ipykernel_58/1074905201.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  [Epoch 1/4] batch 1/1002 | loss 0.7583 | 0.78s/batch | ETA 782s
  [Epoch 1/4] batch 40/1002 | loss 0.8046 | 0.21s/batch | ETA 198s
  [Epoch 1/4] batch 80/1002 | loss 0.7350 | 0.20s/batch | ETA 181s
  [Epoch 1/4] batch 120/1002 | loss 0.7920 | 0.19s/batch | ETA 170s
  [Epoch 1/4] batch 160/1002 | loss 0.7501 | 0.20s/batch | ETA 164s
  [Epoch 1/4] batch 200/1002 | loss 0.6096 | 0.20s/batch | ETA 157s
  [Epoch 1/4] batch 240/1002 | loss 0.5592 | 0.20s/batch | ETA 149s
  [Epoch 1/4] batch 280/1002 | loss 0.7963 | 0.19s/batch | ETA 141s
  [Epoch 1/4] batch 320/1002 | loss 0.5920 | 0.19s/batch | ETA 133s
  [Epoch 1/4] batch 360/1002 | loss 0.7106 | 0.20s/batch | ETA 126s
  [Epoch 1/4] batch 400/1002 | loss 0.6346 | 0.20s/batch | ETA 117s
  [Epoch 1/4] batch 440/1002 | loss 0.7267 | 0.19s/batch | ETA 109s
  [Epoch 1/4] batch 480/1002 | loss 0.5582 | 0.19s/batch | ETA 102s
  [Epoch 1/4] batch 520/1002 | loss 0.4203 | 0.19s/batch | ETA 94s
  [Epoch 1/4] batch 560/1002 | loss 0.5711 | 0.19s/ba

/tmp/ipykernel_58/1074905201.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch  1/4 | train_loss 0.6035 | val_loss 0.4409 | val_F1 0.7566 | val_recall 0.8586 | 206.4s


...

  [Epoch 4/4] batch 40/1002 | loss 0.0942 | 0.19s/batch | ETA 185s
  [Epoch 4/4] batch 80/1002 | loss 0.1414 | 0.19s/batch | ETA 179s
  [Epoch 4/4] batch 120/1002 | loss 0.3508 | 0.19s/batch | ETA 170s
  [Epoch 4/4] batch 160/1002 | loss 0.2987 | 0.20s/batch | ETA 164s
  [Epoch 4/4] batch 200/1002 | loss 0.0191 | 0.20s/batch | ETA 157s
  [Epoch 4/4] batch 240/1002 | loss 0.2213 | 0.19s/batch | ETA 147s
  [Epoch 4/4] batch 280/1002 | loss 0.2642 | 0.19s/batch | ETA 139s
  [Epoch 4/4] batch 320/1002 | loss 0.2661 | 0.19s/batch | ETA 131s
  [Epoch 4/4] batch 360/1002 | loss 0.1908 | 0.19s/batch | ETA 123s
  [Epoch 4/4] batch 400/1002 | loss 0.2416 | 0.19s/batch | ETA 116s
  [Epoch 4/4] batch 440/1002 | loss 0.1026 | 0.19s/batch | ETA 108s
  [Epoch 4/4] batch 480/1002 | loss 0.0720 | 0.19s/batch | ETA 101s
  [Epoch 4/4] batch 520/1002 | loss 0.0988 | 0.19s/batch | ETA 93s
  [Epoch 4/4] batch 560/1002 | loss 0.1899 | 0.19s/batch | ETA 85s
  [Epoch 4/4] batch 600/1002 | loss 0.3112 | 0.19s/b

/tmp/ipykernel_58/1074905201.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch  4/4 | train_loss 0.2016 | val_loss 0.4941 | val_F1 0.7990 | val_recall 0.8237 | 204.7s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

         -> new best, checkpoint saved

Training complete. Best val F1 = 0.7990 at epoch 4


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir_oversampled).to(DEVICE)

_, val_true_os, val_prob_os = predict(model, val_loader, criterion, DEVICE, desc="final val")
_, test_true_os, test_prob_os = predict(model, test_loader, criterion, DEVICE, desc="final test")

print_metrics(evaluate_predictions(val_true_os, val_prob_os, 0.5, "val"))
print_metrics(evaluate_predictions(test_true_os, test_prob_os, 0.5, "test"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_58/1074905201.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


--- val (threshold=0.50) ---
  accuracy  : 0.8465
  precision : 0.7757
  recall    : 0.8237
  f1        : 0.7990
  roc_auc   : 0.9290
  pr_auc    : 0.9084
--- test (threshold=0.50) ---
  accuracy  : 0.8439
  precision : 0.7669
  recall    : 0.8317
  f1        : 0.7980
  roc_auc   : 0.9284
  pr_auc    : 0.9117


In [15]:
for t in [0.5, 0.3, 0.25]:
    print(f"\n=== threshold={t} ===")
    for label, true_arr, prob_arr in [("baseline", test_true_baseline, test_prob_baseline), ("oversampled2x", test_true_os, test_prob_os)]:
        pred = (prob_arr >= t).astype(int)
        mask = (test_df_with_source[SOURCE_COL] == "malicious_prompts").to_numpy()
        malicious_mask = mask & (true_arr == 1)
        recall = pred[malicious_mask].mean()
        fn = int((malicious_mask & (pred == 0)).sum())
        overall = evaluate_predictions(true_arr, prob_arr, t, label)
        print(f"  {label:15s} | malicious_prompts recall={recall:.4f} FN={fn:3d} | overall F1={overall['f1']:.4f} recall={overall['recall']:.4f} precision={overall['precision']:.4f}")


=== threshold=0.5 ===
  baseline        | malicious_prompts recall=0.4397 FN=209 | overall F1=0.8150 recall=0.7681 precision=0.8680
  oversampled2x   | malicious_prompts recall=0.6381 FN=135 | overall F1=0.7980 recall=0.8317 precision=0.7669

=== threshold=0.3 ===
  baseline        | malicious_prompts recall=0.6247 FN=140 | overall F1=0.8000 recall=0.8335 precision=0.7691
  oversampled2x   | malicious_prompts recall=0.7319 FN=100 | overall F1=0.7971 recall=0.8649 precision=0.7392

=== threshold=0.25 ===
  baseline        | malicious_prompts recall=0.6515 FN=130 | overall F1=0.7977 recall=0.8439 precision=0.7562
  oversampled2x   | malicious_prompts recall=0.7507 FN= 93 | overall F1=0.7967 recall=0.8727 precision=0.7328


In [16]:
MAX_EPOCHS_EXTENDED = 8
PATIENCE_EXTENDED = 3   # a bit more patience too, so it can actually find its own stopping point

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_os)  # reuse from the 2x run

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader_oversampled) * MAX_EPOCHS_EXTENDED
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

print(f"Total optimizer steps: {total_steps:,} (warmup: {warmup_steps:,})")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total optimizer steps: 8,016 (warmup: 801)


/tmp/ipykernel_58/737783104.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


In [ ]:
best_val_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0
history_2x_extended = []

checkpoint_dir_2x_extended = os.path.join(SAVE_DIR, "distilbert_oversampled2x_extended_best")

print(f"Starting training — up to {MAX_EPOCHS_EXTENDED} epochs, patience={PATIENCE_EXTENDED}\n")

for epoch in range(1, MAX_EPOCHS_EXTENDED + 1):
    start = time.time()
    train_loss = train_one_epoch(
        model, train_loader_oversampled, criterion, optimizer, scheduler, scaler,
        DEVICE, GRAD_CLIP, epoch, MAX_EPOCHS_EXTENDED
    )
    val_loss, val_true_ex, val_prob_ex = predict(model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch} val")
    val_metrics_ex = evaluate_predictions(val_true_ex, val_prob_ex, 0.5, "val")

    elapsed = time.time() - start
    history_2x_extended.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_f1": val_metrics_ex["f1"], "val_recall": val_metrics_ex["recall"], "seconds": elapsed,
    })

    print(f"Epoch {epoch:>2}/{MAX_EPOCHS_EXTENDED} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
          f"| val_F1 {val_metrics_ex['f1']:.4f} | val_recall {val_metrics_ex['recall']:.4f} | {elapsed:.1f}s")

    if val_metrics_ex["f1"] > best_val_f1:
        best_val_f1 = val_metrics_ex["f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        model.save_pretrained(checkpoint_dir_2x_extended)
        tokenizer.save_pretrained(checkpoint_dir_2x_extended)
        print("         -> new best, checkpoint saved")
    else:
        epochs_without_improvement += 1
        print(f"         -> no improvement ({epochs_without_improvement}/{PATIENCE_EXTENDED})")
        if epochs_without_improvement >= PATIENCE_EXTENDED:
            print(f"Early stopping at epoch {epoch}. Best epoch {best_epoch}.")
            break

print(f"\nTraining complete. Best val F1 = {best_val_f1:.4f} at epoch {best_epoch}")

Starting training — up to 8 epochs, patience=3

  [Epoch 1/8] batch 1/1002 | loss 0.7774 | 0.20s/batch | ETA 198s


/tmp/ipykernel_58/1074905201.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  [Epoch 1/8] batch 40/1002 | loss 0.7532 | 0.19s/batch | ETA 185s
  [Epoch 1/8] batch 80/1002 | loss 0.7913 | 0.20s/batch | ETA 181s
  [Epoch 1/8] batch 120/1002 | loss 0.7696 | 0.20s/batch | ETA 172s
  [Epoch 1/8] batch 160/1002 | loss 0.7418 | 0.20s/batch | ETA 165s
  [Epoch 1/8] batch 200/1002 | loss 0.7550 | 0.19s/batch | ETA 156s
  [Epoch 1/8] batch 240/1002 | loss 0.6237 | 0.19s/batch | ETA 148s
  [Epoch 1/8] batch 280/1002 | loss 0.6500 | 0.19s/batch | ETA 141s
  [Epoch 1/8] batch 320/1002 | loss 0.6292 | 0.19s/batch | ETA 132s
  [Epoch 1/8] batch 360/1002 | loss 0.6696 | 0.19s/batch | ETA 124s
  [Epoch 1/8] batch 400/1002 | loss 0.7485 | 0.19s/batch | ETA 117s
  [Epoch 1/8] batch 440/1002 | loss 0.4922 | 0.19s/batch | ETA 109s
  [Epoch 1/8] batch 480/1002 | loss 0.5454 | 0.19s/batch | ETA 101s
  [Epoch 1/8] batch 520/1002 | loss 0.5910 | 0.19s/batch | ETA 94s
